# Working Memory Test

Calls `manage_tasks` directly with its injected `conversation_id` against a fresh conversation, and reads the stored `task_plan` back from the `sessions` document. Needs Firestore access (the same credentials the gateway uses).

In [1]:
import json
import sys

sys.path.insert(0, "..")

from app.gateway.gateway import create_conversation, fetch_session_memory
from app.orchestrator.tools import manage_tasks

In [2]:
USER_ID = "gateway-test-user"
conversation_id = await create_conversation(USER_ID)
print("conversation_id:", conversation_id)


async def save(**args):
    return await manage_tasks.ainvoke({**args, "conversation_id": conversation_id})


async def show_plan():
    memory = await fetch_session_memory(conversation_id)
    print(json.dumps(memory["task_plan"], indent=1))
    print("history entries (should stay 0):", len(memory["history_messages"]))


def task(text, status="todo"):
    return {"text": text, "status": status}

conversation_id: 4523788f-e773-49b7-85e7-7b4bc33dbc09


## Save a plan

In [8]:
print(await save(
    scope="Model: XGBoost Ranker. Split: test. Bar charts sorted high to low.",
    tasks=[
        task("Reorder rate by department, as a bar chart with a two-sentence takeaway.", "in_progress"),
        task("Top 10 features by SHAP importance, as a bar chart with a short explanation."),
        task("Beeswarm for the most important feature, produce department only."),
    ]))
await show_plan()

Saved the task plan (0 of 3 done).
{
 "tasks": [
  {
   "status": "in_progress",
   "text": "Reorder rate by department, as a bar chart with a two-sentence takeaway."
  },
  {
   "status": "todo",
   "text": "Top 10 features by SHAP importance, as a bar chart with a short explanation."
  },
  {
   "status": "todo",
   "text": "Beeswarm for the most important feature, produce department only."
  }
 ],
 "scope": "Model: XGBoost Ranker. Split: test. Bar charts sorted high to low."
}
history entries (should stay 0): 0


## Update statuses without resending the scope (the scope should stay)

In [4]:
print(await save(tasks=[
    task("Reorder rate by department, as a bar chart with a two-sentence takeaway.", "done"),
    task("Top 10 features by SHAP importance, as a bar chart with a short explanation.", "in_progress"),
    task("Beeswarm for the most important feature, produce department only."),
]))
await show_plan()

Saved the task plan (1 of 3 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart with a two-sentence takeaway."
  },
  {
   "status": "in_progress",
   "text": "Top 10 features by SHAP importance, as a bar chart with a short explanation."
  },
  {
   "status": "todo",
   "text": "Beeswarm for the most important feature, produce department only."
  }
 ],
 "scope": "Model: XGBoost Ranker. Split: test. Bar charts sorted high to low."
}
history entries (should stay 0): 0


## Change the scope, then clear only the scope

In [5]:
tasks = [task("Reorder rate by department, as a bar chart.", "done")]
print(await save(tasks=tasks, scope="Model: LightGBM Classifier. Split: validation."))
await show_plan()
print(await save(tasks=tasks, scope=""))
await show_plan()

Saved the task plan (1 of 1 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart."
  }
 ],
 "scope": "Model: LightGBM Classifier. Split: validation."
}
history entries (should stay 0): 0
Saved the task plan (1 of 1 done).
{
 "tasks": [
  {
   "status": "done",
   "text": "Reorder rate by department, as a bar chart."
  }
 ],
 "scope": ""
}
history entries (should stay 0): 0


## Argument errors (each should come back as a readable message)

In [6]:
print(await save(tasks=[task("A", "in_progress"), task("B", "in_progress")]))
print(await save(tasks=[task("   ")]))
print(await save(tasks=[task("x" * 501)]))
print(await save(tasks=[task(f"Task {n}") for n in range(31)]))
print(await save(tasks=[task("A")], scope="s" * 801))

1 validation error for ManageTasksArgs
tasks
  Value error, More than one task is in progress; mark only the one you are working on. [type=value_error, input_value=[{'text': 'A', 'status': ...status': 'in_progress'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, Task 1 is blank. [type=value_error, input_value=[{'text': '   ', 'status': 'todo'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, Task 1 is 501 characters; keep each under 500. [type=value_error, input_value=[{'text': 'xxxxxxxxxxxxxx...xxx', 'status': 'todo'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for ManageTasksArgs
tasks
  Value error, A task list holds at most 30 tasks; merge or drop some. [type=value_error, input_value=[{'t

## Clear the whole plan

In [7]:
print(await save(tasks=[]))
await show_plan()

Cleared the task plan.
{}
history entries (should stay 0): 0
